# Explore EIA/FERC utility entity matching with Splink

This notebook loads the FERC and EIA utility tables from a local copy plopped into `devtools/`, inspects the available columns, builds a normalized set of candidate match fields, and uses Splink to explore how the records line up.

This is just a draft, obvious improvements are:
- reading in the pickles from dagster storage or (eventually) reading in the parquet files
- more robust match-specific cleaning
- ideally we'd like `utility_id_ferc1` and `report_year` to be a unique key for the FERC dataframe. right now it's not, which means we're doing the match with a lot of near duplicate records or records that could be consolidated by filling nulls, which isn't great.

The main goals are:
- inspect the available columns in both tables
- focus on `name`, `year`, `street_address`, `city`, `state`, and `zip`
- surface other possible identifiers that might help matching
- compare the two FERC address columns to see which one is more useful for linkage
- generate candidate utility matches with Splink for manual review


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from dagster import AssetKey
from pudl.etl import defs

import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.blocking_analysis import count_comparisons_from_blocking_rule
from splink.exploratory import completeness_chart, profile_columns

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# DEVTOOLS_DIR = Path.cwd()
# FERC_PATH = DEVTOOLS_DIR / "core_ferc1__yearly_identification_certification.pkl"
# EIA_PATH = DEVTOOLS_DIR / "out_eia__yearly_utilities.parquet"

# ferc_raw = pd.read_pickle(FERC_PATH)
# eia_raw = pd.read_parquet(EIA_PATH)

In [ ]:
ferc_raw = defs.load_asset_value(AssetKey("core_ferc1__yearly_identification_certification"))
eia_raw = defs.load_asset_value(AssetKey("out_eia__yearly_utilities"))

In [ ]:
ferc_raw.shape, eia_raw.shape

Let's see what potential columns are available on each side to match on.

In [ ]:
COLUMN_GROUPS = {
    "name", "date", "year", "address", "city", "zip", "state", "id"
}

def get_cols_in_group(df, group_str):
    return [col for col in df if group_str in col]

candidate_cols = {}
for group in COLUMN_GROUPS:
    candidate_cols[group] = {}
    candidate_cols[group]["ferc"] = get_cols_in_group(ferc_raw, group)
    candidate_cols[group]["eia"] = get_cols_in_group(eia_raw, group)

In [ ]:
pd.DataFrame(candidate_cols)

# Investigate candidate match columns

Now we can narrow down the columns of interest and take a look at some of the candidate match columns. We want to check for their nullness, the uniqueness of values, any obvious cleaning needs, etc.

In [ ]:
ferc_candidate_cols = [
    "utility_id_ferc1",
    "office_state",
    "contact_state",
    "utility_name_ferc1",
    "contact_name",
    "office_zip",
    "contact_zip",
    "filing_date",
    "report_year",
    "office_street_address",
    "contact_address",
    "office_city",
    "contact_city"
]
eia_candidate_cols = [
    "utility_id_eia",
    "state",
    "utility_name_eia",
    "contact_firstname",
    "contact_lastname",
    "contact_firstname_2",
    "contact_lastname_2",
    "report_date",
    "zip_code",
    "street_address",
    "address_2",
    "city"
]

In [ ]:
ferc_raw[ferc_candidate_cols].info()

In [ ]:
eia_raw[eia_candidate_cols].info()

In [ ]:
ferc_raw[ferc_candidate_cols].isnull().sum()/len(ferc_raw)

In [ ]:
eia_raw[eia_candidate_cols].isnull().sum()/len(eia_raw)

Ok we can already tell that contact name won't be good to match on since there's lots of nulls in EIA. So far it seems like the following will be good:
- utility name
- address (likely office but maybe contact address if it matches better with EIA address)
- zip code (same question of office vs contact)
- city

We'll use report date / year for something called blocking, which means only record pairs with matching report years will be considered.

In [ ]:
eia_candidate_cols = [col for col in eia_candidate_cols if col not in ["contact_firstname", "contact_lastname", "contact_firstname_2", "contact_lastname_2"]]
ferc_candidate_cols = [col for col in ferc_candidate_cols if col not in ["contact_name", "filing_date"]]

## Profile Columns

We have an idea of nullness, let's learn more about the candidate columns by analyzing the distribution of values in the columns using a built in splink functionality. This is important because (taken from [splink docs](https://moj-analytical-services.github.io/splink/demos/tutorials/02_Exploratory_analysis.html)):

```
The distribution of values in your data is important for two main reasons:

Columns with higher cardinality (number of distinct values) are usually more useful for data linking. For instance, date of birth is a much stronger linkage variable than gender.

The skew of values is important. If you have a city column that has 1,000 distinct values, but 75% of them are London, this is much less useful for linkage than if the 1,000 values were equally distributed
```

In [ ]:
eia_profile_cols = [col for col in eia_candidate_cols if col not in ["utility_id_eia", "report_date"]]

profile_columns(eia_raw[eia_profile_cols], db_api=DuckDBAPI(), top_n=10, bottom_n=5)

In [ ]:
ferc_profile_cols = [col for col in ferc_candidate_cols if col not in ["utility_id_ferc1", "report_year"]]

profile_columns(ferc_raw[ferc_profile_cols], db_api=DuckDBAPI(), top_n=10, bottom_n=5)

There's definitely some weird skew with the most sampled values on addresses, cities, and zips on both the EIA and FERC side... but nothing too worrying at this point.

# Select match columns and rename

Lets take an initial stab at selecting some columns for matching. For now I'm going to go with the office address and city over the contact address and city.

In [ ]:
match_cols = {
    "utility_name": {"eia": "utility_name_eia", "ferc": "utility_name_ferc1"},
    "street_address": {"eia": "street_address", "ferc": "office_street_address"},
    "city": {"eia": "city", "ferc": "office_city"},
    "zip": {"eia": "zip_code", "ferc": "office_zip"},
    "state": {"eia": "state", "ferc": "office_state"}
}

In [ ]:
eia_df = eia_raw.copy()
ferc_df = ferc_raw.copy()
for col, rename_dict in match_cols.items():
    eia_df = eia_df.rename(columns={rename_dict["eia"]: col})
    ferc_df = ferc_df.rename(columns={rename_dict["ferc"]: col})

In [ ]:
eia_df["report_year"] = pd.to_datetime(eia_df["report_date"]).dt.year

# Clean for matching

Let's do some basic cleaning to get the dataframes ready for matching. Splink suggests:

* unique IDs for records
* standardizing date formats, matching text case, and handling invalid data
* Ensure null values (or other 'not known' indicators) are represented as true nulls, not empty strings

In [ ]:
eia_df[["utility_id_eia", "report_year"]].duplicated().value_counts()

In [ ]:
ferc_df[["utility_id_ferc1", "report_year"]].duplicated().value_counts()

In [ ]:
ferc_df[ferc_df[["utility_id_ferc1", "report_year"]].duplicated(keep=False)].sort_values(by=["utility_id_ferc1", "report_year"])

In [ ]:
ferc_df.duplicated().value_counts()

In [ ]:
# TODO: fill the null values in the utility_id_ferc1, report_year groups and then count duplicates created by this ffill, something like:
# filled_df = ferc_df.groupby(["utility_id_ferc1", "report_year"], group_keys=False).ffill()

Perform some basic cleaning

In [ ]:
match_cols

In [ ]:
def clean_for_matching(df, dataset_name: str):
    df["source_dataset"] = dataset_name
    df["record_id"] = df.index.astype("string")
    for str_col in ["utility_name", "street_address", "city", "zip", "state"]:
        df[str_col] = df[str_col].astype("string").str.strip().str.lower().replace("", pd.NA)
    return df

In [ ]:
eia_df = clean_for_matching(eia_df, "eia")

In [ ]:
ferc_df = clean_for_matching(ferc_df, "ferc")

## Side quest to verify that office address is better than contact address on the ferc side

In [ ]:
ferc_df_contact_address = ferc_df.copy()
ferc_df_contact_address["street_address"] = ferc_df_contact_address["contact_address"].astype("string").str.strip().str.lower().replace("", pd.NA)

In [ ]:
def address_option_summary(ferc_df: pd.DataFrame, eia_df: pd.DataFrame, street_col: str) -> dict[str, object]:
    ferc_non_null = ferc_df[ferc_df["street_address"].notna()].copy()
    eia_non_null = eia_df[eia_df["street_address"].notna()].copy()

    full_address_keys = ["report_year", "street_address", "city", "state", "zip"]
    loose_address_keys = ["street_address", "city", "state"]
    name_state_keys = ["report_year", "utility_name", "state"]

    exact_full = ferc_non_null[full_address_keys].merge(
        eia_non_null[full_address_keys].drop_duplicates(),
        how="inner",
        on=full_address_keys,
    )
    exact_loose = ferc_non_null[loose_address_keys].merge(
        eia_non_null[loose_address_keys].drop_duplicates(),
        how="inner",
        on=loose_address_keys,
    )
    name_state = ferc_df[name_state_keys].merge(
        eia_df[name_state_keys].drop_duplicates(),
        how="inner",
        on=name_state_keys,
    )

    return {
        "ferc_street_column": street_col,
        "ferc_records": len(ferc_df),
        "ferc_non_null_street": int(ferc_df["street_address"].notna().sum()),
        "ferc_street_fill_pct": round(100 * ferc_df["street_address"].notna().mean(), 1),
        "ferc_unique_streets": int(ferc_df["street_address"].nunique(dropna=True)),
        "exact_full_address_overlap": len(exact_full),
        "exact_loose_address_overlap": len(exact_loose),
        "name_state_overlap": len(name_state),
    }


ferc_match_frames = {
    "office_address": ferc_df,
    "contact_address": ferc_df_contact_address
}

address_option_df = pd.DataFrame(
    [
        address_option_summary(ferc_df, eia_df, street_col)
        for street_col, ferc_df in ferc_match_frames.items()
    ]
).sort_values(
    ["exact_full_address_overlap", "exact_loose_address_overlap", "ferc_street_fill_pct"],
    ascending=False,
)

display(address_option_df)


Okay, office_address is the better address to go with, as we suspected.

# Before splinking, check for exact matches with a merge

In [ ]:
ferc_df

In [ ]:
quick_exact_candidates = (
    ferc_df.merge(
        eia_df,
        how="inner",
        on=["report_year", "state"],
        suffixes=("_ferc", "_eia"),
    )
    .assign(
        same_name=lambda df: df["utility_name_ferc"].fillna("").eq(df["utility_name_eia"].fillna("")),
        same_street=lambda df: df["street_address_ferc"].fillna("").eq(df["street_address_eia"].fillna("")),
        same_city=lambda df: df["city_ferc"].fillna("").eq(df["city_eia"].fillna("")),
        same_zip=lambda df: df["zip_ferc"].fillna("").eq(df["zip_eia"].fillna("")),
    )
)

quick_exact_candidates["agreement_score"] = quick_exact_candidates[
    ["same_name", "same_street", "same_city", "same_zip"]
].sum(axis=1)

quick_exact_candidates = quick_exact_candidates.sort_values(
    ["agreement_score", "same_name", "same_street", "same_zip"],
    ascending=False,
)

display(
    quick_exact_candidates[
        [
            "record_id_ferc",
            "utility_name_ferc",
            "street_address_ferc",
            "city_ferc",
            "state",
            "zip_ferc",
            "record_id_eia",
            "utility_name_eia",
            "street_address_eia",
            "city_eia",
            "zip_eia",
            "agreement_score",
            "same_name",
            "same_street",
            "same_city",
            "same_zip",
        ]
    ].head(50)
)


In [ ]:
len(quick_exact_candidates)

In [ ]:
len(quick_exact_candidates.utility_id_eia.unique())/len(eia_df.utility_id_eia.unique())

In [ ]:
len(quick_exact_candidates.utility_id_ferc1.unique())/len(ferc_df.utility_id_ferc1.unique())

Yay there are a lot of exact match candidates with a lot of coverage of utility IDs. We could stop here... or we could splink it to get higher coverage.

# Splink it!

We start by defining these comparison rules/thresholds for each match column. We also define "blocking rules" which say which pairs we should even consider as potential matches. Let's start by testing how big of a "candidate pair set" we create with each proposed blocking rule.

In [ ]:
match_cols.keys()

In [ ]:
cols_to_keep = list(match_cols.keys()) + ["record_id", "report_year"]
eia_match_df = eia_df[cols_to_keep]
ferc_match_df = ferc_df[cols_to_keep]

In [ ]:
comparison_name = cl.NameComparison(
    "utility_name",
    jaro_winkler_thresholds=[0.97, 0.93, 0.88],
)
comparison_street = cl.NameComparison(
    "street_address",
    jaro_winkler_thresholds=[0.97, 0.92, 0.85],
)
comparison_city = cl.NameComparison(
    "city",
    jaro_winkler_thresholds=[0.95, 0.88],
)
comparison_state = cl.ExactMatch("state")
comparison_zip = cl.ExactMatch("zip")
comparison_year = cl.ExactMatch("report_year")

comparison_state.configure(term_frequency_adjustments=True)
comparison_zip.configure(term_frequency_adjustments=True)

blocking_rules = [
    block_on("report_year", "state"),
    block_on("report_year", "zip"),
]

for blocking_rule in blocking_rules:
    display(Markdown(f"### Blocking rule\n`{blocking_rule}`"))
    display(
        count_comparisons_from_blocking_rule(
            table_or_tables=[eia_match_df, ferc_match_df],
            blocking_rule=blocking_rule,
            link_type="link_only",
            unique_id_column_name="record_id",
            db_api=DuckDBAPI(),
        )
    )


## Set model parameters

In [ ]:
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="record_id",
    comparisons=[
        comparison_name,
        comparison_year,
        comparison_street,
        comparison_city,
        comparison_state,
        comparison_zip,
    ],
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
    probability_two_random_records_match=1 / max(len(eia_match_df), 1),
)

linker = Linker(
    [eia_match_df, ferc_match_df],
    settings=settings,
    input_table_aliases=["eia", "ferc"],
    db_api=DuckDBAPI(),
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("report_year", "state")
)

predictions = linker.inference.predict(threshold_match_probability=0.1)
preds_df = predictions.as_pandas_dataframe().sort_values("match_probability", ascending=False)

candidate_review = preds_df.merge(
    eia_match_df.add_suffix("_eia"),
    left_on="record_id_l",
    right_on="record_id_eia",
    how="left",
).merge(
    ferc_match_df.add_suffix("_ferc"),
    left_on="record_id_r",
    right_on="record_id_ferc",
    how="left",
)

review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",
    "gamma_utility_name",
    "gamma_street_address",
    "gamma_city",
    "gamma_state",
    "gamma_zip",
    "gamma_report_year",
]

display(candidate_review[review_columns].head(100))


In [ ]:
preds = candidate_review[candidate_review.match_probability > .99]

In [ ]:
preds[(preds.match_probability<.9999) & (preds.match_probability>.995)]

## Next steps

A few fields beyond the core address/name/year set may also help if they exist in the source tables:
- utility or respondent IDs
- EIN or other tax identifiers
- phone numbers
- service territory or state of incorporation
- website or contact metadata

Those can be added as exact-match or high-weight comparison columns once the basic utility/address linkage is done.